In [12]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy
import copy

from scipy.sparse import coo_matrix, block_diag, identity, hstack, csr_matrix, csc_matrix, vstack
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import time 
from time import perf_counter
import matplotlib as mpl

from pyiga import assemble, bspline, vform, geometry, vis, solvers, utils, topology, ieti, algebra, operators, adaptive
from pyiga import algebra_cy, ieti_cy, bspline_cy

from scipy.sparse.linalg import aslinearoperator as LinOp

np.set_printoptions(linewidth=100000)
np.set_printoptions(precision=5)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
def Inductor(deg,N, airgap=0.025):
    kvs=42*[2*(bspline.make_knots(deg,0.0,1.0,N),)]
    
    geos=[      
        geometry.unit_square().scale((0.5)).translate((-0.5,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((0,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((0.25,-0.5)),
        geometry.unit_square().scale((0.5,0.5)).translate((0.5,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.,-0.5)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.25,-0.5)),
        geometry.unit_square().scale(0.5).translate((1.5,-0.5)),
        
        geometry.unit_square().scale((0.5,0.25)).translate((-0.5,0)),
        geometry.unit_square().scale(0.25),
        geometry.unit_square().scale(0.25).translate((0.25,0)),
        geometry.unit_square().scale((0.5,0.25)).translate((0.5,0)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.,0)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.25,0)),
        geometry.unit_square().scale((0.5,0.25)).translate((1.5,0)),
        
        geometry.unit_square().scale((0.5,airgap)).translate((-0.5,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((0,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((0.25,0.25)),
        geometry.unit_square().scale((0.5,airgap)).translate((0.5,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((1.,0.25)),
        geometry.unit_square().scale((0.25,airgap)).translate((1.25,0.25)),
        geometry.unit_square().scale((0.5,airgap)).translate((1.5,0.25)),
        
        geometry.unit_square().scale((0.5,0.5)).translate((-0.5,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0.25,0.25+airgap)),
        geometry.unit_square().scale((0.5,0.5)).translate((0.5,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.,0.25+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.25,0.25+airgap)),
        geometry.unit_square().scale((0.5,0.5)).translate((1.5,0.25+airgap)),
        
        geometry.unit_square().scale((0.5,0.25)).translate((-0.5,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((0,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((0.25,0.75+airgap)),
        geometry.unit_square().scale((0.5,0.25)).translate((0.5,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.,0.75+airgap)),
        geometry.unit_square().scale((0.25,0.25)).translate((1.25,0.75+airgap)),
        geometry.unit_square().scale((0.5,0.25)).translate((1.5,0.75+airgap)),
        
        geometry.unit_square().scale(0.5).translate((-0.5,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((0.25,1.0+airgap)),
        geometry.unit_square().scale((0.5,0.5)).translate((0.5,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.,1.0+airgap)),
        geometry.unit_square().scale((0.25,0.5)).translate((1.25,1.0+airgap)),
        geometry.unit_square().scale(0.5).translate((1.5,1.0+airgap)),
         ]
    patches=list(zip(kvs,geos))
    M = topology.MultiPatch(patches)
    M.rename_domain(0,'Air')
    M.set_domain_id({'Fe':{8,9,10,11,12,22,24,26,29,30,31,32,33}, 'C1':{23}, 'C2':{25}})
    M.h_refine({i:1 for i in range(14,21)});                                        #split airgap patches further in y-axis to make up for anisotropy 
    M.h_refine({i:1 for i in list(range(14,21))+list(range(42,49))});               #split airgap patches further in y-axis to make up for anisotropy 
    #M.h_refine({i:1 for i in list(range(14,21))+list(range(42,49))+list(range(49,63))}); #split airgap patches further in y-axis to make up for anisotropy
    return M

In [16]:
M = Inductor(5,20)
MB = assemble.MultiBasis(M, subspace='C0')

setting up constraints took 0.11828899383544922 seconds.
Basis setup took 0.004553556442260742 seconds


In [54]:
dir_bcs = MB.set_fixed_boundary({0:0})
Kh = MB.assemble_volume('a*inner(grad(u),grad(v)) * dx', a=1e10, arity=2)
Fh = MB.assemble_volume('f * v * dx', arity=1, f=1)
LS = assemble.RestrictedLinearSystem(Kh,Fh,dir_bcs)

In [55]:
A = LS.A.copy()
from sksparse.cholmod import cholesky
t0 = perf_counter()
factor = cholesky(A.tocsc())
#print(factor.L().nnz)
t1 = perf_counter()

x = factor.solve_A(LS.b)
t2 = perf_counter()
print(np.linalg.norm(LS.A@x-LS.b))

del factor 
print("factor", t1-t0)
print("solve", t2-t1)

9.636319129336722e-14
factor 0.21011740000039936
solve 0.00984249999964959


In [56]:
import pyMKL

A = LS.A
t0 = perf_counter()
solver = pyMKL.pardisoSolver(A, mtype=-2)
solver.factor()
t1 = perf_counter()

x = solver.solve(LS.b)
t2 = perf_counter()
print(np.linalg.norm(LS.A@x-LS.b))

solver.clear()
print("factor", t1-t0)
print("solve", t2-t1)

8.263242718678057e-14
factor 0.33144880000008925
solve 0.02722330000005968


In [59]:
from pymklpardiso import PardisoSolver
#A = scipy.sparse.csr_matrix(scipy.sparse.triu(LS.A, format='csr'))
#A.sort_indices()
A = LS.A

t0 = perf_counter()
solver = PardisoSolver(A, mtype=11)
t1 = perf_counter()

x = solver.solve(LS.b)
t2 = perf_counter()
print(np.linalg.norm(LS.A@x-LS.b))

#solver.clear()
print("factor", t1-t0)
print("solve", t2-t1)

9.263037957997925e-14
factor 0.2840214999996533
solve 0.01103140000031999


In [42]:
t = time.time()
operators.make_solver(LS.A.tocsc(), spd=True)
print(time.time()-t)

2.2262959480285645


In [32]:
import sksparse.cholmod
print(sksparse.cholmod.__file__)

/home/wolfman/miniforge3/envs/sci/lib/python3.12/site-packages/sksparse/cholmod.cpython-312-x86_64-linux-gnu.so


In [9]:
%whos

Variable       Type                          Data/Info
------------------------------------------------------
A              csr_matrix                    <Compressed Sparse Row sp<...>41672)	1.6348003848004846
Fh             ndarray                       443857: 443857 elems, type `float64`, 3550856 bytes (3.3863601684570312 Mb)
Inductor       function                      <function Inductor at 0x7f80082c2a20>
Kh             csc_matrix                    <Compressed Sparse Column<...>)	-3.1725797135722214e-08
LS             RestrictedLinearSystem        <pyiga.assemble.Restricte<...>object at 0x7f7fa9cdab10>
LinOp          function                      <function aslinearoperator at 0x7f7fe01868e0>
M              MultiPatch                    <pyiga.topology.MultiPatc<...>object at 0x7f7fa8644a10>
MB             MultiBasis                    <pyiga.assemble.MultiBasi<...>object at 0x7f80082ac750>
adaptive       module                        <module 'pyiga.adaptive' <...>pyiga/pyiga/ada

In [10]:
print("A:", A.data.nbytes/1024**3,
      A.indices.nbytes/1024**3,
      A.indptr.nbytes/1024**3)

print("Kh:", Kh.data.nbytes/1024**3,
      Kh.indices.nbytes/1024**3,
      Kh.indptr.nbytes/1024**3)

A: 0.38034821301698685 0.19017410650849342 0.0016453638672828674
Kh: 0.3822721317410469 0.19113606587052345 0.0016534999012947083
